# IBM AML Dataset — Initial Data Exploration

**Purpose:** Inspect the raw CSV files, validate the schema, understand class imbalance,
and produce summary statistics to guide graph construction decisions.

This notebook directly informs SQ1 (graph construction design decisions).

In [ ]:
import sys
from pathlib import Path

# Make src/ importable
sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

from src.data.loader import load_raw_data
from src.utils.config import DataConfig

%load_ext autoreload
%autoreload 2

In [ ]:
# --- Choose dataset variant ---
# Replace with "HI-Small" or your chosen variant
cfg = DataConfig(
    raw_dir="../data/raw",
    dataset_variant="LI-Small",
)

In [ ]:
# --- Load raw data ---
accounts, transactions = load_raw_data(cfg)

In [ ]:
# --- Accounts: shape & columns ---
print(f"Accounts: {accounts.shape}")
accounts.head(10)

In [ ]:
# --- Transactions: shape & columns ---
print(f"Transactions: {transactions.shape}")
transactions.head(10)

In [ ]:
# --- Class imbalance ---
label_counts = transactions["is_laundering"].value_counts()
print("Laundering vs Non-laundering:")
print(label_counts)
print(f"\nLaundering ratio: {label_counts[1] / len(transactions):.6f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
label_counts.plot.bar(ax=axes[0], color=["steelblue", "crimson"])
axes[0].set_title("Edge Label Distribution")
axes[0].set_ylabel("Count")
label_counts.plot.pie(ax=axes[1], labels=["Non-laundering", "Laundering"],
                       autopct="%1.4f%%", colors=["steelblue", "crimson"])
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# --- Transaction amounts ---
if "amount" in transactions.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Raw amounts
    axes[0].hist(transactions["amount"], bins=100, edgecolor="none", alpha=0.7)
    axes[0].set_title("Transaction Amount (raw)")
    axes[0].set_xlabel("Amount")
    
    # Log-transformed
    axes[1].hist(np.log1p(transactions["amount"]), bins=100, edgecolor="none", alpha=0.7, color="darkgreen")
    axes[1].set_title("Transaction Amount (log₁+x)")
    axes[1].set_xlabel("log₁ₚ(Amount)")
    
    plt.tight_layout()
    plt.show()
    
    print(transactions["amount"].describe())

In [ ]:
# --- Payment format distribution ---
if "payment_format" in transactions.columns:
    print("Payment format distribution:")
    print(transactions["payment_format"].value_counts())
    
    transactions["payment_format"].value_counts().plot.barh(color="steelblue")
    plt.title("Payment Format Distribution")
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Account activity (degree) ---
degree_out = transactions.groupby("from_account").size()
degree_in = transactions.groupby("to_account").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(degree_out, bins=100, edgecolor="none", alpha=0.7)
axes[0].set_title("Out-degree Distribution")
axes[0].set_xlabel("Number of outgoing transactions")
axes[0].set_ylabel("Number of accounts")

axes[1].hist(degree_in, bins=100, edgecolor="none", alpha=0.7, color="darkgreen")
axes[1].set_title("In-degree Distribution")
axes[1].set_xlabel("Number of incoming transactions")

plt.tight_layout()
plt.show()

print(f"Max out-degree: {degree_out.max()}, Max in-degree: {degree_in.max()}")
print(f"Mean out-degree: {degree_out.mean():.2f}, Mean in-degree: {degree_in.mean():.2f}")

In [ ]:
# --- Check account overlap between accounts.csv and transactions ---
acct_ids = set(accounts["account_id"])
txn_senders = set(transactions["from_account"])
txn_receivers = set(transactions["to_account"])
txn_ids = txn_senders | txn_receivers

print(f"Unique accounts in accounts.csv:   {len(acct_ids)}")
print(f"Unique accounts in transactions:   {len(txn_ids)}")
print(f"Orphan transactions (not in accounts.csv): {len(txn_ids - acct_ids)}")
print(f"Accounts without any transactions: {len(acct_ids - txn_ids)}")

In [ ]:
# --- Temporal span ---
if "timestamp" in transactions.columns:
    ts = pd.to_datetime(transactions["timestamp"], unit="s", errors="coerce")
    if ts.isna().any():
        ts = pd.to_datetime(transactions["timestamp"], errors="coerce")
    print(f"Time range: {ts.min()}  →  {ts.max()}")
    print(f"Duration: {(ts.max() - ts.min()).days} days")
    
    # Transaction volume over time
    ts_sorted = ts.sort_values()
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(ts_sorted, np.arange(len(ts_sorted)) / len(ts_sorted), lw=1)
    ax.set_xlabel("Time")
    ax.set_ylabel("Cumulative fraction of transactions")
    ax.set_title("Cumulative Transaction Distribution Over Time")
    plt.tight_layout()
    plt.show()

## Observations

Record key findings here to inform graph construction decisions:
- Class imbalance ratio:
- Feature availability:
- Degree distribution characteristics:
- Temporal span and snapshot sizing considerations: